# Lesson 1 · NumPy Foundations: Arrays, Indexing & Slicing
### The engine room of every data-science library you will ever use

| | |
|---|---|
| **Type** | LESSON 01 |
| **Time** | about 90 minutes |
| **Level** | Beginner |
| **Prerequisites** | Basic Python (variables, lists, loops, functions) |

**By the end of this notebook you will be able to:**

- Explain *why* NumPy exists and demonstrate its speed advantage
- Create 1-D, 2-D and 3-D arrays and read their `shape`, `ndim`, `size` and `dtype`
- Pick out any element, row, column or block with indexing and slicing
- Filter data with boolean masks and `np.where`
- Reshape, transpose, stack and concatenate arrays
- Avoid the #1 NumPy trap: views vs copies

## 🛠️ Setup
Run this cell first. It imports the tools and loads the data every problem below relies on.

In [1]:

import sys; sys.path.insert(0, "..")          # lets the notebook find course_utils.py
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check

# --- Data used by the problems -------------------------------------------------
rng = np.random.default_rng(42)
weekly_prices = np.round(rng.normal(100, 15, size=20), 1)       # 20 weeks of tomato prices (KES/kg)
market_grid   = np.round(rng.normal(60, 8, size=(3, 8)), 1)     # 3 markets x 8 weeks
df = pd.read_csv(DATA_DIR / "market_prices_clean.csv")           # real(ish) market data, loaded properly in Lesson 3
print("Setup complete ✔   weekly_prices has", weekly_prices.size, "values")


Setup complete ✔   weekly_prices has 20 values


## 👋 Welcome

Hello, and welcome to the first real lesson! I'm going to teach you the way I wish someone had taught me:
**one idea at a time, always with a picture in your head, and always with something you can run.**

Here's the deal for this notebook:

1. I explain an idea in plain language.
2. I show it working.
3. I ask you to *predict* what happens before I run it. (This is the single best learning trick I know.)
4. You solve problems. Hints are hidden; try for **ten honest minutes** before opening one.

> 🎯 **Today's story.** You are a data analyst at an agricultural market information service in Nairobi. Every Monday
> morning, enumerators record the price of maize, beans, tomatoes and more at markets across Kenya. Your boss asks:
> *"Which market is cheapest this week? Which prices look suspicious? Can you do this for 15,000 records by lunchtime?"*
> Spreadsheets will creak. **NumPy** will not.

## 1 · Why NumPy? (Seeing is believing)

Python lists are wonderfully flexible: they can hold numbers, text, other lists… anything. But that flexibility costs
speed, because Python has to check *what kind of thing* each element is every time you touch it.

A **NumPy array** (`ndarray`) makes a trade: *every element must be the same type*, stored side by side in memory.
In return, NumPy can hand whole arrays to blazing-fast compiled code.

Let me prove it. We'll square one million numbers two ways.

In [2]:
import time

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

t0 = time.perf_counter()
py_squares = [x * x for x in py_list]            # Python loop
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
np_squares = np_arr * np_arr                      # NumPy: one operation on the whole array
t_np = time.perf_counter() - t0

print(f"Python list : {t_list*1000:8.1f} ms")
print(f"NumPy array : {t_np*1000:8.2f} ms")
print(f"NumPy was roughly {t_list / t_np:.0f}x faster")

Python list :     38.3 ms
NumPy array :     4.89 ms
NumPy was roughly 8x faster


> 💡 **Intuition.** Think of a Python list as a shelf of labelled boxes, each box possibly holding something different. A NumPy array is a neat egg-tray: identical slots, packed tightly, so a machine can process a whole tray at once.

> 🤖 **Why machine learning cares.** Machine-learning datasets are *tables of numbers*. Training on millions of rows would be impossible with Python loops. Every ML library (scikit-learn, PyTorch, TensorFlow) is built on NumPy-style arrays.

## 2 · Creating arrays

The most direct way is `np.array(...)` from a list. Let's record five weekly maize prices (KES per kg):

In [3]:
maize = np.array([52.0, 55.5, 54.0, 61.2, 58.9])
print(maize)
print("type :", type(maize))
print("dtype:", maize.dtype)

[52.  55.5 54.  61.2 58.9]
type : <class 'numpy.ndarray'>
dtype: float64


Every array carries four facts about itself. Memorise these, you will print them *constantly*:

| Attribute | Meaning | For `maize` |
|---|---|---|
| `.shape` | size along each dimension (a tuple) | `(5,)` |
| `.ndim` | number of dimensions | `1` |
| `.size` | total number of elements | `5` |
| `.dtype` | the single data type of all elements | `float64` |

In [4]:
print("shape:", maize.shape, "| ndim:", maize.ndim, "| size:", maize.size, "| dtype:", maize.dtype)

shape: (5,) | ndim: 1 | size: 5 | dtype: float64


### Creation shortcuts you'll use every day

In [5]:
print("zeros   :", np.zeros(4))                       # placeholders to fill later
print("ones    :", np.ones((2, 3)))                    # 2 rows x 3 columns of 1.0
print("full    :", np.full(3, 99.9))                   # constant value
print("arange  :", np.arange(0, 10, 2))               # like range(), but returns an array: start, stop(excluded), step
print("linspace:", np.linspace(0, 1, 5))              # 5 evenly spaced points, endpoints INCLUDED
print("eye     :\n", np.eye(3))                        # identity matrix

zeros   : [0. 0. 0. 0.]
ones    : [[1. 1. 1.]
 [1. 1. 1.]]
full    : [99.9 99.9 99.9]
arange  : [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
eye     :
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


> ⚠️ **Common mistake.** `np.arange(0, 1, 0.1)` can surprise you with floating-point rounding about how many items you get. When you care about the *count* of points, use `np.linspace(start, stop, count)`.

### Data types (`dtype`): one array, one type

What do you think this prints? Decide first.

In [6]:
mixed = np.array([1, 2.5, 3])
print(mixed, mixed.dtype)                 # ints get "promoted" to floats

surprise = np.array([1, 2.5, "maize"])
print(surprise, surprise.dtype)           # everything becomes text!

[1.  2.5 3. ] float64
['1' '2.5' 'maize'] <U32


NumPy never keeps a *mix*: it upcasts everything to the one type that can hold all values. Convert deliberately with `.astype()`:

In [7]:
prices_text = np.array(["52.5", "55.0", "61.2"])
prices_num = prices_text.astype(float)
print(prices_num, prices_num.dtype)
print("Whole shillings:", prices_num.astype(int))        # astype(int) TRUNCATES (cuts decimals), it does not round
print("Rounded        :", np.round(prices_num).astype(int))

[52.5 55.  61.2] float64
Whole shillings: [52 55 61]
Rounded        : [52 55 61]


> 📝 **Note.** Missing values: NumPy represents 'not a number' as `np.nan`, which is a *float*. So an integer array can't hold NaN: one more reason pandas is so popular for messy data.

## 3 · Multidimensional arrays

A table is just a 2-D array: **rows × columns**. Let's store maize prices for **3 markets** (rows) over **4 weeks** (columns).

In [8]:
maize_grid = np.array([[48.0, 49.5, 51.0, 50.2],     # market 0: Eldoret
                       [52.0, 53.5, 55.0, 54.1],     # market 1: Nairobi
                       [45.5, 46.0, 47.2, 46.8]])    # market 2: Kisumu
print(maize_grid)
print("shape:", maize_grid.shape, "(3 rows = markets, 4 columns = weeks)")

[[48.  49.5 51.  50.2]
 [52.  53.5 55.  54.1]
 [45.5 46.  47.2 46.8]]
shape: (3, 4) (3 rows = markets, 4 columns = weeks)


### The most important idea in NumPy: **axes**

```
                axis 1  →  (across the columns = weeks)
              wk0   wk1   wk2   wk3
axis 0   mkt0 [48.0  49.5  51.0  50.2]
   ↓     mkt1 [52.0  53.5  55.0  54.1]
         mkt2 [45.5  46.0  47.2  46.8]
```

When you reduce along an axis (sum, mean, max…), **that axis disappears**:

* `axis=0` → collapse the rows → you get **one value per column** (per week).
* `axis=1` → collapse the columns → you get **one value per row** (per market).

In [9]:
print("Average price each WEEK   (axis=0):", maize_grid.mean(axis=0).round(2))
print("Average price each MARKET (axis=1):", maize_grid.mean(axis=1).round(2))
print("Overall average          (no axis):", maize_grid.mean().round(2))

Average price each WEEK   (axis=0): [48.5  49.67 51.07 50.37]
Average price each MARKET (axis=1): [49.68 53.65 46.38]
Overall average          (no axis): 49.9


> 🧠 **Pause and predict.** What will `maize_grid.max(axis=1)` return, and what shape will it have? (Answer: the best price seen by each market, shape `(3,)`.)

**Three dimensions.** Add a *crop* dimension: `(crops, markets, weeks)`.

In [10]:
beans_grid = np.round(maize_grid * 2.6, 1)                 # pretend beans cost 2.6x maize
cube = np.stack([maize_grid, beans_grid])                   # stack along a NEW first axis
print("cube shape:", cube.shape, "  -> (crops, markets, weeks)")
print("Beans, market 2, week 0 :", cube[1, 2, 0])

cube shape: (2, 3, 4)   -> (crops, markets, weeks)
Beans, market 2, week 0 : 118.3


## 4 · Indexing & slicing: pointing at exactly the data you want

**Rule 1: Counting starts at 0.** The first element is index `0`, the last is `-1`.

**Rule 2: Slices are `start:stop:step`, and `stop` is *excluded*.** `a[2:5]` gives positions 2, 3, 4.

In [11]:
x = np.arange(10, 20)        # [10 11 12 ... 19]
print("x           :", x)
print("x[0], x[-1] :", x[0], x[-1])
print("x[2:5]      :", x[2:5])            # positions 2,3,4
print("x[:3]       :", x[:3])             # first three
print("x[-3:]      :", x[-3:])            # last three
print("x[::2]      :", x[::2])            # every second element
print("x[::-1]     :", x[::-1])           # reversed

x           : [10 11 12 13 14 15 16 17 18 19]
x[0], x[-1] : 10 19
x[2:5]      : [12 13 14]
x[:3]       : [10 11 12]
x[-3:]      : [17 18 19]
x[::2]      : [10 12 14 16 18]
x[::-1]     : [19 18 17 16 15 14 13 12 11 10]


**In 2-D, give one index per axis, separated by commas: `a[row, column]`.** A colon `:` means "everything along this axis".

In [12]:
print("one cell        maize_grid[1, 2]    =", maize_grid[1, 2])
print("one row         maize_grid[1]       =", maize_grid[1])
print("one column      maize_grid[:, 0]    =", maize_grid[:, 0])
print("a block         maize_grid[0:2, 1:3]=\n", maize_grid[0:2, 1:3])
print("last col, all rows maize_grid[:, -1]=", maize_grid[:, -1])

one cell        maize_grid[1, 2]    = 55.0
one row         maize_grid[1]       = [52.  53.5 55.  54.1]
one column      maize_grid[:, 0]    = [48.  52.  45.5]
a block         maize_grid[0:2, 1:3]=
 [[49.5 51. ]
 [53.5 55. ]]
last col, all rows maize_grid[:, -1]= [50.2 54.1 46.8]


> 🧠 **Pause and predict.** What is the **shape** of `maize_grid[:, 0]`? Of `maize_grid[:, 0:1]`? (Answer: `(3,)` and `(3, 1)`. Slicing with a range keeps the dimension; a single integer drops it.)

In [13]:
print(maize_grid[:, 0].shape, maize_grid[:, 0:1].shape)

(3,) (3, 1)


### ⚠️ The #1 NumPy trap: slices are **views**, not copies

A slice does **not** duplicate the data, instead it gives you a *window* onto the same memory. Change the window and you change the original.

In [14]:
demo = weekly_prices.copy()          # work on a copy so the lesson data stays safe
window = demo[:3]                    # a VIEW of the first 3 values
window[0] = 9999                     # modify the view...
print("demo[0] is now:", demo[0])    # ...and the original changed too!

safe = demo[:3].copy()               # .copy() makes an independent array
safe[0] = -1
print("demo[0] still :", demo[0])

demo[0] is now: 9999.0
demo[0] still : 9999.0


> 💡 **Intuition.** A view is like a window in a wall: paint the glass and you've painted what the room sees. `.copy()` is a photograph: draw on the photo all you like; the room is untouched. Rule of thumb: **if you plan to modify a slice, copy it first.**

## 5 · Boolean masks: filtering by condition

This is the technique you'll use most in data analysis. A comparison on an array gives an array of `True/False`  (a **mask**)
and you can use the mask to *select* elements.

In [15]:
prices = weekly_prices
mask = prices > 110
print("first 8 prices:", prices[:8])
print("first 8 mask  :", mask[:8])
print("How many weeks above 110? ", mask.sum(), "  (True counts as 1)")
print("Those prices:", prices[mask])
print("Share of weeks above 110:", mask.mean().round(2), "  (mean of booleans = proportion!)")

first 8 prices: [104.6  84.4 111.3 114.1  70.7  80.5 101.9  95.3]
first 8 mask  : [False False  True  True False False False False]
How many weeks above 110?  6   (True counts as 1)
Those prices: [111.3 114.1 113.2 111.7 116.9 113.2]
Share of weeks above 110: 0.3   (mean of booleans = proportion!)


Combine conditions with `&` (and), `|` (or), `~` (not)  **with parentheses around each condition**.

In [16]:
mid_range = prices[(prices >= 90) & (prices <= 110)]
extremes  = prices[(prices < 80) | (prices > 120)]
print("In the 90-110 band:", mid_range)
print("Extreme weeks     :", extremes)

In the 90-110 band: [104.6 101.9  95.3  99.7 101.  107.  105.5  99.3]
Extreme weeks     : [70.7]


> ⚠️ **Common mistake.** Never write `prices > 90 and prices < 110` on arrays; Python's `and` can't compare whole arrays and raises *“truth value of an array is ambiguous”*. Use `&` and `|`, and don't forget the parentheses: `&` binds tighter than `>`.

**`np.where`** has two personalities:

* `np.where(condition)` → the **positions** where the condition holds.
* `np.where(condition, a, b)` → pick `a` where True, else `b` (a vectorised if/else).

In [17]:
positions = np.where(prices > 115)[0]
print("Week numbers above 115:", positions)

labels = np.where(prices > 100, "expensive", "affordable")
print(labels[:6])

print("argmax (position of the highest price):", prices.argmax(), "->", prices.max())
print("argmin (position of the lowest price) :", prices.argmin(), "->", prices.min())

Week numbers above 115: [13]
['expensive' 'affordable' 'expensive' 'expensive' 'affordable'
 'affordable']
argmax (position of the highest price): 13 -> 116.9
argmin (position of the lowest price) : 4 -> 70.7


**Fancy indexing** pass a *list of positions* to pull specific elements, in any order:

In [18]:
print(prices[[0, 5, 10]])
order = np.argsort(prices)                # positions that would sort the array
print("Three cheapest weeks:", order[:3], "-> prices", prices[order[:3]])

[104.6  80.5 113.2]
Three cheapest weeks: [4 5 1] -> prices [70.7 80.5 84.4]


## 6 · Reshaping, transposing, stacking

In [19]:
a = np.arange(12)
print(a)
print(a.reshape(3, 4))             # 12 values -> 3 rows x 4 columns
print(a.reshape(2, -1).shape)      # -1 means "figure this dimension out for me"  -> (2, 6)
print(a.reshape(3, 4).T.shape)     # .T = transpose (swap rows and columns)     -> (4, 3)

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
(4, 3)


In [20]:
w1 = np.array([10, 11, 12]); w2 = np.array([20, 21, 22])
print("vstack (stack as rows)   :\n", np.vstack([w1, w2]))
print("hstack (side by side)    :", np.hstack([w1, w2]))
print("concatenate on new data  :", np.concatenate([w1, w2]))
print("column vector via newaxis:", w1[:, np.newaxis].shape, " <- we'll need this for broadcasting in Lesson 2")

vstack (stack as rows)   :
 [[10 11 12]
 [20 21 22]]
hstack (side by side)    : [10 11 12 20 21 22]
concatenate on new data  : [10 11 12 20 21 22]
column vector via newaxis: (3, 1)  <- we'll need this for broadcasting in Lesson 2


> 📝 **Note.** `reshape` needs the new shape to hold exactly the same number of elements: 12 values can become 3×4, 2×6, 4×3, 2×2×3… but never 5×3. `reshape` usually returns a *view* too.

## 7 · From a pandas table to a NumPy array (a sneak peek at real data)

Real data arrive as files. We'll learn pandas properly in Lesson 3, but here's the bridge: a pandas column becomes a NumPy
array with `.to_numpy()`.

In [21]:
mask_w = (df["crop"] == "Maize") & (df["market"] == "Wakulima Market")
maize_wakulima = df.loc[mask_w, "price_kes_per_kg"].to_numpy()

print("shape  :", maize_wakulima.shape, "-> one price per week for 4 years")
print("first 5:", maize_wakulima[:5])
print("cheapest week index:", maize_wakulima.argmin(), "price:", maize_wakulima.min())
print("dearest  week index:", maize_wakulima.argmax(), "price:", maize_wakulima.max())
print("Price rose from", maize_wakulima[0], "to", maize_wakulima[-1], "over the period")

shape  : (209,) -> one price per week for 4 years
first 5: [44.8 48.1 47.8 49.1 48.2]
cheapest week index: 0 price: 44.8
dearest  week index: 113 price: 81.0
Price rose from 44.8 to 59.3 over the period


---
# 🧑‍🏫 Worked examples

*Study these before attempting the problems. Notice the **thinking steps**, not just the code.*

## Worked example 1: "Which weeks should the trader worry about?"

> **Question.** Using `weekly_prices`, flag the weeks whose price is more than 10% above the average and report how many there are,
> along with the *week numbers* and the worst week.

**Thinking steps**
1. Compute the benchmark: `avg = prices.mean()`.
2. Define "worry": `prices > 1.10 * avg` → a Boolean mask.
3. Count with `.sum()`, locate with `np.where`, find the worst with `argmax`.

In [22]:
avg = weekly_prices.mean()
worry = weekly_prices > 1.10 * avg
print(f"Average price: {avg:.1f}  |  worry threshold: {1.10*avg:.1f}")
print("Number of worrying weeks:", worry.sum())
print("Week numbers            :", np.where(worry)[0])
print("Worst week              :", weekly_prices.argmax(), "at", weekly_prices.max(), "KES/kg")

Average price: 99.5  |  worry threshold: 109.5
Number of worrying weeks: 6
Week numbers            : [ 2  3 10 11 13 18]
Worst week              : 13 at 116.9 KES/kg


## Worked example 2: "Which market is cheapest each week?"

> **Question.** `market_grid` has 3 markets (rows) × 8 weeks (columns). For *each week*, name the cheapest market.

**Thinking steps**
1. "For each week" = for each **column** → we want to look *down* the rows → `axis=0`.
2. `argmin(axis=0)` gives the winning row index per column.
3. Use the indices to look up names (fancy indexing).

In [23]:
names = np.array(["Eldoret", "Nairobi", "Kisumu"])
winner = market_grid.argmin(axis=0)
print("Winning row index per week:", winner)
print("Cheapest market per week  :", names[winner])
vals, counts = np.unique(names[winner], return_counts=True)
print("How often does each market win?", {str(v): int(c) for v, c in zip(vals, counts)})

Winning row index per week: [0 2 2 1 1 1 2 2]
Cheapest market per week  : ['Eldoret' 'Kisumu' 'Kisumu' 'Nairobi' 'Nairobi' 'Nairobi' 'Kisumu'
 'Kisumu']
How often does each market win? {'Eldoret': 1, 'Kisumu': 4, 'Nairobi': 3}


---
# 🏋️ Problems

Rules of the game:

* Every problem uses variables from the **Setup** cell (`weekly_prices`, `market_grid`, `df`).
* Store your result in `answer`, run the cell, and read the feedback.
* Stuck? Open **Hint 1**. Still stuck after five more minutes? **Hint 2**. The last hint is a strong nudge.
* Compare with the `solutions/` notebook *only after* you've finished (or truly given up for the day).

### 🧩 Problem 1.1: Count the expensive weeks

How many weeks in `weekly_prices` had a price **strictly above 110** KES/kg?

*Expected type:* an integer.

<details>
<summary>💡 Hint 1</summary>

A comparison like `weekly_prices > 110` produces an array of True/False values.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`True` behaves as 1 and `False` as 0 when you add them. Which method adds up an array?

</details>

*Auto-check id:* `01-p1`  ·  *Set the variable `answer`, then run the cell.*

In [24]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p1', answer)

⏳ 01-p1: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 1.2: Newest first

Return the **last five** prices of `weekly_prices`, ordered from **newest to oldest** (so the final week comes first).

*Expected type:* an array of 5 numbers.

<details>
<summary>💡 Hint 1</summary>

The last five elements: use a negative start in the slice.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

To reverse, add a step of `-1`. Slice first, then reverse: `something[-5:][::-1]`.

</details>

*Auto-check id:* `01-p2`  ·  *Set the variable `answer`, then run the cell.*

In [25]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p2', answer)

⏳ 01-p2: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 1.3: Cheapest market, every week

`market_grid` holds prices for 3 markets (rows `0, 1, 2`) over 8 weeks (columns). Return, for each week, the **row index** of the cheapest market.

*Expected type:* an array of 8 integers, each 0, 1 or 2.

<details>
<summary>💡 Hint 1</summary>

You want one answer per *column*. Which axis do you collapse to get one value per column?

</details>

<details>
<summary>💡 Hint 2</summary>

There's a NumPy function that returns the *position* of the minimum, not the minimum itself.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`market_grid.argmin(axis=0)`

</details>

*Auto-check id:* `01-p3`  ·  *Set the variable `answer`, then run the cell.*

In [26]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p3', answer)

⏳ 01-p3: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 1.4: Tame the outliers

A data-entry clerk sometimes fat-fingers prices. Treat any price **greater than 1.25 × the median** of `weekly_prices` as an error and replace it with the median. Return the **sum** of the cleaned array, rounded to 1 decimal.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

Compute `med = np.median(weekly_prices)` first.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.where(condition, value_if_true, value_if_false)` builds the cleaned array without a loop.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`cleaned = np.where(weekly_prices > 1.25*med, med, weekly_prices)`, then sum and round.

</details>

*Auto-check id:* `01-p4`  ·  *Set the variable `answer`, then run the cell.*

In [27]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p4', answer)

⏳ 01-p4: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 1.5: A slice through the cube

Create `cube = np.arange(24).reshape(2, 3, 4)`. Think of it as 2 crops × 3 markets × 4 weeks. Return `cube[1, :, 2]`: the values for crop 1, **all** markets, week 2.

*Before you run it,* predict the shape and the three numbers on paper. *Expected type:* an array of 3 numbers.

<details>
<summary>💡 Hint 1</summary>

Fill the 24 numbers row by row: crop 0 holds 0–11, crop 1 holds 12–23.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Inside crop 1, each market is a row of 4 weeks: market 0 = 12,13,14,15 … so week 2 of market 0 is 14.

</details>

*Auto-check id:* `01-p5`  ·  *Set the variable `answer`, then run the cell.*

In [28]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p5', answer)

⏳ 01-p5: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 1.6: How often was Kongowea tomato above its own average?

Using the real data in `df`, extract the **Tomatoes** prices at **Kongowea Market** (as a NumPy array, in date order) and count the weeks in which the price was **above that series' own mean**.

*Expected type:* an integer.

<details>
<summary>💡 Hint 1</summary>

Build a Boolean mask with two conditions joined by `&` (remember the parentheses!), then use `.loc[mask, column].to_numpy()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`series.mean()` is the benchmark. A new comparison + `.sum()` counts the weeks.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

```python
m = (df["crop"] == "Tomatoes") & (df["market"] == "Kongowea Market")
s = df.loc[m, "price_kes_per_kg"].to_numpy()
```

</details>

*Auto-check id:* `01-p6`  ·  *Set the variable `answer`, then run the cell.*

In [29]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('01-p6', answer)

⏳ 01-p6: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

## ✅ Check your understanding

Answer in your head (or out loud!) first, *then* open the answer.

**Q1. Why does `np.array([1, 2.5, 'a'])` not contain numbers?**

<details>
<summary>Show answer</summary>

An array has **one** dtype. NumPy upcasts to the type that can hold everything: here, text (`<U32`). Convert with `.astype(float)` when the data are numeric strings.

</details>

**Q2. `A` has shape `(3, 4)`. What is the shape of `A[:, 0]`, and of `A.mean(axis=1)`?**

<details>
<summary>Show answer</summary>

`A[:, 0]` is a single column → `(3,)`. `A.mean(axis=1)` collapses the columns, leaving one value per row → `(3,)`.

</details>

**Q3. You write `b = a[:3]; b[0] = 0`. Did `a` change? How do you prevent it?**

<details>
<summary>Show answer</summary>

Yes, slices are *views*. Use `b = a[:3].copy()` for an independent array.

</details>

**Q4. What does `mask.mean()` give you when `mask` is a Boolean array?**

<details>
<summary>Show answer</summary>

The **proportion** of `True` values (because True=1, False=0).

</details>

**Q5. Why do we write `(a > 1) & (a < 5)` instead of `a > 1 and a < 5`?**

<details>
<summary>Show answer</summary>

`and` demands a single True/False; arrays hold many. `&` works element-wise, and the parentheses are needed because `&` has higher precedence than comparisons.

</details>

## 🧾 Cheat-sheet

| I want to… | Code |
|---|---|
| Make an array | `np.array([..])`, `np.zeros(n)`, `np.arange(a,b,step)`, `np.linspace(a,b,n)` |
| Inspect it | `a.shape`, `a.ndim`, `a.size`, `a.dtype` |
| Element / row / column | `a[i, j]`, `a[i]`, `a[:, j]` |
| A block | `a[r0:r1, c0:c1]` |
| Every 2nd, reversed | `a[::2]`, `a[::-1]` |
| Filter | `a[a > 5]`, `a[(a > 1) & (a < 9)]` |
| Positions | `np.where(cond)[0]`, `a.argmax()`, `a.argmin(axis=0)` |
| If/else | `np.where(cond, x, y)` |
| Reshape | `a.reshape(r, c)`, `a.reshape(-1)`, `a.T` |
| Join | `np.vstack`, `np.hstack`, `np.concatenate`, `np.stack` |
| Safe copy | `a[:3].copy()` |

## ➡️ Next up
**Lesson 2** adds the superpowers: *vectorisation*, *broadcasting*, *matrix maths* and *random numbers*.